# IsaacLab-Arena Workshop — follow-along notebook
**Simulation Workflow → Setup: Isaac Lab-Arena** (NVIDIA GR00T end-to-end workflow), running on NVIDIA Brev.

This notebook runs on the **host** of your Brev instance (JupyterLab at port 8888). Steps that the NVIDIA page runs *inside* the Arena container are executed here through `docker exec`, so you can follow along without leaving the notebook. You can equally type the same commands in the browser desktop terminal (see the participant PDF).

Reference page: https://docs.nvidia.com/learning/physical-ai/gr00t-e2e-workflow/latest/simulation-workflow/sim-setup-isaac-lab-arena.html

## 0. Machine check
Confirm the GPU, Docker with the NVIDIA runtime, and the browser desktop are all up.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!docker info 2>/dev/null | grep -E "Runtimes|Default Runtime"
!ss -tlnp | grep -E ":6080" || echo "noVNC not listening — ask the organizer to run: sudo systemctl restart gpu-desktop"
!DISPLAY=:0 glxinfo 2>/dev/null | grep "OpenGL renderer" || echo "desktop not up"

## Step 1–2. The repository (already cloned)
The workflow clones `IsaacLab-Arena` at `release/0.2.1` with submodules. On Brev this was done over HTTPS, so no GitHub SSH key is needed. Verify the branch and that both submodules are checked out (no leading `-`).

In [ ]:
%cd ~/IsaacLab-Arena
!git branch --show-current
!git submodule status

If a submodule line starts with `-`, run the next cell once.

In [ ]:
!git config --global url."https://github.com/".insteadOf "git@github.com:"
!git submodule update --init --recursive
!git submodule status

## Step 3. Host directories
Mounted into the container as `/datasets`, `/models`, `/eval`.

In [ ]:
!mkdir -p $HOME/datasets $HOME/models $HOME/eval && ls -d $HOME/datasets $HOME/models $HOME/eval

## Step 4. Launch the Docker container
`./docker/run_docker.sh` is interactive (`-it`), so from a notebook we start the same container **detached** using the script's own settings, then talk to it with `docker exec`. The image `isaaclab_arena:latest` is pre-built by the Brev setup script.

If you prefer, run `cd ~/IsaacLab-Arena && ./docker/run_docker.sh` in the desktop terminal instead; the container name is the same (`isaaclab_arena-latest`) so the later cells still work.

In [ ]:
import os, subprocess, shlex
home = os.path.expanduser("~")
name = "isaaclab_arena-latest"
running = subprocess.run(["docker","ps","-q","-f",f"name=^{name}$"],capture_output=True,text=True).stdout.strip()
if running:
    print("container already running:", name)
else:
    subprocess.run(["docker","rm","-f",name],capture_output=True)
    cmd = f'''docker run -d --name {name} --runtime=nvidia --gpus=all --privileged --ipc=host --net=host \
      --ulimit memlock=-1 --ulimit stack=-1 \
      -e DISPLAY=:0 -e ACCEPT_EULA=Y -e PRIVACY_CONSENT=Y \
      -e ISAACLAB_PATH=/workspaces/isaaclab_arena/submodules/IsaacLab \
      -v {home}/IsaacLab-Arena:/workspaces/isaaclab_arena \
      -v {home}/datasets:/datasets -v {home}/models:/models -v {home}/eval:/eval \
      -v /tmp/.X11-unix:/tmp/.X11-unix \
      -w /workspaces/isaaclab_arena isaaclab_arena:latest sleep infinity'''
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(out.stdout or out.stderr)
!docker ps --filter name=isaaclab_arena-latest --format "{{.Names}}  {{.Status}}"

Sanity check inside the container: GPU visible and `DISPLAY=:0` set.

In [ ]:
!docker exec isaaclab_arena-latest bash -lc 'nvidia-smi -L; echo DISPLAY=$DISPLAY; python -c "import isaaclab, isaaclab_arena; print(\"isaaclab + isaaclab_arena import OK\")"'

## Step 5 (optional, 20–30 min). Full test suite
Skip during the live session. Uncomment to run later.

In [ ]:
# !docker exec isaaclab_arena-latest bash -lc '/isaac-sim/python.sh -m pytest -sv -m "with_cameras and not with_subprocess" isaaclab_arena/tests/'
# !docker exec isaaclab_arena-latest bash -lc '/isaac-sim/python.sh -m pytest -sv -m "not with_cameras and not with_subprocess" isaaclab_arena/tests/'
# !docker exec isaaclab_arena-latest bash -lc '/isaac-sim/python.sh -m pytest -sv -m with_subprocess isaaclab_arena/tests/'

## Step 6. Tutorial directories inside the container

In [ ]:
!docker exec isaaclab_arena-latest bash -lc 'export DATASET_DIR=/datasets/isaaclab_arena/static_apple_tutorial; mkdir -p $DATASET_DIR; export MODELS_DIR=/models/isaaclab_arena/static_apple_tutorial; mkdir -p $MODELS_DIR; ls -d $DATASET_DIR $MODELS_DIR'

## Step 7–8. Validate the environment with automated tests (~2 min)
Runs the G1 static apple-to-plate environment headlessly. Expected tail: `2 passed, ... warnings in ~95s`.

If the apple falls through the shelf on the very first execution, simply re-run this cell (documented first-run effect).

In [ ]:
!docker exec isaaclab_arena-latest bash -lc '/isaac-sim/python.sh -m pytest isaaclab_arena/tests/test_g1_static_pick_and_place.py -v 2>&1 | tail -25'

## Bonus. Watch an Arena environment in the GUI
This launches the policy runner with the Kit visualizer **in the background** (`zero_action` policy, so the robot just holds still). Runner options go *before* the environment name, environment options after it. The Isaac Lab window appears on your browser desktop (the port-6080 Secure Link) after 3–5 minutes on the first launch, about a minute afterwards.

Note: through `docker exec` there is no `python` alias, so cells call Isaac Sim's interpreter `/isaac-sim/python.sh` directly. In the desktop terminal inside the container, plain `python` works. Orbit with right-drag, pan with middle-drag, Shift + left-drag to nudge objects.

In [ ]:
!docker exec -d isaaclab_arena-latest bash -lc '/isaac-sim/python.sh isaaclab_arena/evaluation/policy_runner.py --policy_type zero_action --viz kit --num_steps 3000 pick_and_place_maple_table --embodiment droid_rel_joint_pos --pick_up_object rubiks_cube_hot3d_robolab --destination_location bowl_ycb_robolab --hdr home_office_robolab > /tmp/env_runner.log 2>&1'
print("launched — switch to the desktop tab; first launch takes 3-5 min. Log: /tmp/env_runner.log inside the container")

Try a variation: different background HDR and objects. Stop the previous one first.

In [ ]:
!docker exec isaaclab_arena-latest bash -lc 'pkill -f policy_runner.py' ; sleep 3
!docker exec -d isaaclab_arena-latest bash -lc '/isaac-sim/python.sh isaaclab_arena/evaluation/policy_runner.py --policy_type zero_action --viz kit --num_steps 3000 pick_and_place_maple_table --embodiment droid_rel_joint_pos --pick_up_object mustard_bottle_hot3d_robolab --destination_location wooden_bowl_hot3d_robolab --hdr billiard_hall_robolab > /tmp/env_runner.log 2>&1'

In [ ]:
# stop the GUI runner when done
!docker exec isaaclab_arena-latest bash -lc 'pkill -f policy_runner.py; tail -5 /tmp/env_runner.log'

## Step 9–12. Isaac GR00T outside the container
The workflow keeps GR00T separate from Arena so it can be updated independently. The setup script already cloned it at the pinned commit and ran `uv sync`. Verify:

In [ ]:
import os
os.environ["ISAAC_GR00T_DIR"] = os.path.expanduser("~/Isaac-GR00T")
!cd $ISAAC_GR00T_DIR && git rev-parse HEAD && echo "expected: 4b1dca9d88d2a0b9ea5a65aa61c82ff89f5c4f0e"
!ls -d $ISAAC_GR00T_DIR/.venv 2>/dev/null && ~/.local/bin/uv --version || echo "uv env missing — run the next cell"

Only if the check above failed, do it by hand exactly as the workflow page says:

In [ ]:
# !git clone https://github.com/NVIDIA/Isaac-GR00T.git $ISAAC_GR00T_DIR
# !cd $ISAAC_GR00T_DIR && git checkout 4b1dca9d88d2a0b9ea5a65aa61c82ff89f5c4f0e
# !cd $ISAAC_GR00T_DIR && ~/.local/bin/uv sync

## Checkpoint
- [x] Arena container running with GPU and display
- [x] `test_g1_static_pick_and_place.py` passing
- [x] Isaac Sim GUI verified in the browser
- [x] Isaac GR00T environment ready

Next pages of the workflow: **Sim Environment Code Review → Sim Teleop and WBC → Sim Data Export → GR00T Fine-Tuning on Sim Data → Sim Evaluation**
https://docs.nvidia.com/learning/physical-ai/gr00t-e2e-workflow/latest/simulation-workflow/

## Cleanup (end of session)

In [ ]:
!docker exec isaaclab_arena-latest bash -lc 'pkill -f policy_runner.py' 2>/dev/null
!docker stop isaaclab_arena-latest